In [1]:
import duckdb
import pandas as pd

conn = duckdb.connect('/home/jassiel/Documentos/Labs/repo-principal/lab-10/base_clinica.duckdb')

In [2]:
#Laboratorio 11: Analítica longitudinal
#Actividad 1: de anidado a legible

In [3]:
#Pacientes diabéticos sin prueba de HbA1c
#Original
conn.execute("""
SELECT DISTINCT
    p.Id AS patient_id,
    p.FIRST AS nombre,
    p.LAST AS apellido
FROM patients p
JOIN conditions c ON p.Id = c.PATIENT
WHERE LOWER(c.DESCRIPTION) LIKE '%diabetes%'
  AND NOT EXISTS (
      SELECT 1 
      FROM observations o 
      WHERE o.PATIENT = p.Id 
        AND (LOWER(o.DESCRIPTION) LIKE '%hba1c%' OR LOWER(o.DESCRIPTION) LIKE '%hemoglobin a1c%')
  );
""").df()

,patient_id,nombre,apellido
0,cbe9f5e1-165a-9507-d904-f0fa486ea1c5,Rex53,Murphy561
1,5b6a19a6-e5fb-44d4-4646-5f9daf911e17,Raymundo71,Tromp100
2,4b2cc620-5e18-f943-cf43-212e55047402,Zachary28,Rolfson709
3,aa08743c-9520-8b20-5072-e6b40de9e5aa,Leo278,Walter473
4,5bb9328e-56f5-a14c-7fdf-3911be4be7bc,Benton624,Harris789
5,4358663f-ee77-411b-cf3f-f2f5ee07c18d,Max124,Ziemann98
6,3cb59357-5d6e-bb70-e7d2-2b16ee0357d6,Roberto515,Santiago500
7,3584c5f8-85da-a91a-6e7e-5df8753189e0,Jerry654,Orn563
8,b9c35397-c3a9-bc5b-5dc0-818f5d5f0944,Cherise743,Rodriguez71
9,07c8d48c-adae-5755-6f73-5768743fcde4,Fran41,Ledner144


In [4]:
#CTE
conn.execute("""
WITH pacientes_diabeticos AS (
    SELECT DISTINCT p.Id, p.FIRST, p.LAST
    FROM patients p
    JOIN conditions c ON p.Id = c.PATIENT
    WHERE LOWER(c.DESCRIPTION) LIKE '%diabetes%'
),
mediciones_hba1c AS (
    SELECT DISTINCT PATIENT 
    FROM observations 
    WHERE LOWER(DESCRIPTION) LIKE '%hba1c%' OR LOWER(DESCRIPTION) LIKE '%hemoglobin a1c%'
)
SELECT pd.*
FROM pacientes_diabeticos pd
LEFT JOIN mediciones_hba1c mh ON pd.Id = mh.PATIENT
WHERE mh.PATIENT IS NULL;
""").df()

,Id,FIRST,LAST
0,3cb59357-5d6e-bb70-e7d2-2b16ee0357d6,Roberto515,Santiago500
1,07c8d48c-adae-5755-6f73-5768743fcde4,Fran41,Ledner144
2,5b6a19a6-e5fb-44d4-4646-5f9daf911e17,Raymundo71,Tromp100
3,4b2cc620-5e18-f943-cf43-212e55047402,Zachary28,Rolfson709
4,3584c5f8-85da-a91a-6e7e-5df8753189e0,Jerry654,Orn563
5,b9c35397-c3a9-bc5b-5dc0-818f5d5f0944,Cherise743,Rodriguez71
6,cbe9f5e1-165a-9507-d904-f0fa486ea1c5,Rex53,Murphy561
7,aa08743c-9520-8b20-5072-e6b40de9e5aa,Leo278,Walter473
8,5bb9328e-56f5-a14c-7fdf-3911be4be7bc,Benton624,Harris789
9,4358663f-ee77-411b-cf3f-f2f5ee07c18d,Max124,Ziemann98


Ganamos modularidad y facilidad de depuración: aislar la cohorte y las mediciones en bloques separados permite probar cada CTE individualmente antes del cruce, haciendo que la lógica del anti-join sea evidente a simple vista.

In [5]:
#Promedio de tiempo de seguimiento clínico
#Original
conn.execute("""
SELECT 
    ROUND(AVG(dias_seguimiento), 2) AS promedio_dias_historia
FROM (
    SELECT 
        PATIENT,
        DATEDIFF('day', MIN(START), MAX(START)) AS dias_seguimiento
    FROM encounters
    GROUP BY PATIENT
) sub;
""").df()

,promedio_dias_historia
0,11072.31


In [7]:
#CTE
conn.execute("""
WITH historia_pacientes AS (
    SELECT 
        PATIENT,
        DATEDIFF('day', MIN(START), MAX(START)) AS dias_seguimiento
    FROM encounters
    GROUP BY PATIENT
)
SELECT 
    ROUND(AVG(dias_seguimiento), 2) AS promedio_dias_historia
FROM historia_pacientes;
""").df()

,promedio_dias_historia
0,11072.31


Pasamos de leer el código de adentro hacia afuera a una lectura lineal de arriba hacia abajo. En este caso particular de una sola anidación la ganancia de rendimiento es nula, pero prepara el terreno visual para encadenar más pasos sin colapsar la legibilidad.

In [8]:
#Actividad 2: depurar por partes
#Bloque 1: Inspeccionar la primera CTE aislada
print("Bloque 1: pacientes_diabeticos")
df_cte1 = conn.execute("""
    SELECT DISTINCT p.Id, p.FIRST, p.LAST
    FROM patients p
    JOIN conditions c ON p.Id = c.PATIENT
    WHERE LOWER(c.DESCRIPTION) LIKE '%diabetes%'
""").df()
print(f"Filas devueltas: {len(df_cte1)}")
display(df_cte1.head(3))

#Bloque 2: inspeccionar la segunda CTE aislada
print("\nBloque 2: mediciones_hba1c")
df_cte2 = conn.execute("""
    SELECT DISTINCT PATIENT 
    FROM observations 
    WHERE LOWER(DESCRIPTION) LIKE '%hba1c%' OR LOWER(DESCRIPTION) LIKE '%hemoglobin a1c%'
""").df()
print(f"Filas devueltas: {len(df_cte2)}")
display(df_cte2.head(3))

#Bloque 3: el cruce final completo
print("\nBloque 3: Resultado final (anti-join)")
df_final = conn.execute("""
    WITH pacientes_diabeticos AS (
        SELECT DISTINCT p.Id, p.FIRST, p.LAST
        FROM patients p
        JOIN conditions c ON p.Id = c.PATIENT
        WHERE LOWER(c.DESCRIPTION) LIKE '%diabetes%'
    ),
    mediciones_hba1c AS (
        SELECT DISTINCT PATIENT 
        FROM observations 
        WHERE LOWER(DESCRIPTION) LIKE '%hba1c%' OR LOWER(DESCRIPTION) LIKE '%hemoglobin a1c%'
    )
    SELECT pd.*
    FROM pacientes_diabeticos pd
    LEFT JOIN mediciones_hba1c mh ON pd.Id = mh.PATIENT
    WHERE mh.PATIENT IS NULL;
""").df()
print(f"Filas devueltas: {len(df_final)}")
display(df_final.head(3))

Bloque 1: pacientes_diabeticos
Filas devueltas: 227


,Id,FIRST,LAST
0,6f2da7ee-8c88-43d2-adb8-42d6dd761f99,Tawanna466,Leuschke194
1,8b876a1d-0e0d-8087-9f34-6391ddb05bd5,Elly836,Braun514
2,d9c992c1-c095-f114-4974-69458b775c9a,Arturo47,Kreiger457



Bloque 2: mediciones_hba1c
Filas devueltas: 233


,PATIENT
0,291036fc-6c6a-0bc2-f7a5-f65bbace9d1c
1,7f4bd2d3-ddda-962a-aad1-84e4091e7a0f
2,225bb2cd-1075-877e-3ff2-b99bf6791845



Bloque 3: Resultado final (anti-join)
Filas devueltas: 10


,Id,FIRST,LAST
0,5b6a19a6-e5fb-44d4-4646-5f9daf911e17,Raymundo71,Tromp100
1,4b2cc620-5e18-f943-cf43-212e55047402,Zachary28,Rolfson709
2,3584c5f8-85da-a91a-6e7e-5df8753189e0,Jerry654,Orn563


Resultados de la inspección:
Bloque 1 (pacientes_diabeticos): 227 filas devueltas.
Bloque 2 (mediciones_hba1c): 233 filas devueltas.
Bloque 3 (cruce final): 10 filas devueltas (los pacientes que están en el Bloque 1 pero no en el Bloque 2).

¿Qué ventaja práctica se demostró?
Al utilizar CTEs, el código actúa como un pipeline de transformación de datos. Si el resultado final de 10 pacientes me pareciera extraño, puedo simplemente ejecutar `SELECT *` del primer o segundo bloque de forma independiente para auditar si el error viene de un filtro mal escrito en los diagnósticos o en los laboratorios. En una consulta anidada tradicional, esto es imposible sin desarmar y reescribir todo el bloque de código desde cero.

In [9]:
#Actividad 3: primera y última visita por paciente
print("1. Solución con GROUP BY")
df_groupby = conn.execute("""
    SELECT 
        PATIENT,
        MIN(START) AS primera_visita,
        MAX(START) AS ultima_visita,
        COUNT(Id) AS total_visitas
    FROM encounters
    GROUP BY PATIENT
    ORDER BY PATIENT;
""").df()
display(df_groupby.head(5))


print("\n2. Solución con Window Functions")
df_windows = conn.execute("""
    SELECT DISTINCT
        PATIENT,
        FIRST_VALUE(START) OVER (PARTITION BY PATIENT ORDER BY START) AS primera_visita,
        -- LA TRAMPA: Si no ponemos la ventana completa, LAST_VALUE devuelve la fila actual
        LAST_VALUE(START) OVER (
            PARTITION BY PATIENT 
            ORDER BY START 
            RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS ultima_visita,
        COUNT(Id) OVER (PARTITION BY PATIENT) AS total_visitas
    FROM encounters
    ORDER BY PATIENT;
""").df()
display(df_windows.head(5))

#Verificación de que dan exactamente lo mismo
son_iguales = df_groupby.equals(df_windows)
print(f"\n¿Los resultados son exactamente iguales? {son_iguales}")

1. Solución con GROUP BY


,PATIENT,primera_visita,ultima_visita,total_visitas
0,00dbff13-8767-26c9-5315-d938617e3e07,2016-05-15 21:30:33,2026-06-07 21:30:33,36
1,012acae3-d0f3-3493-b3cb-bb430437d0f8,2019-11-30 07:49:09,2025-11-15 07:49:09,18
2,0276003c-ce0c-a4cb-b2ed-5443d03cdf46,1997-02-20 03:02:44,2025-01-30 03:02:44,16
3,032e8472-885d-73e5-f9cf-e9505c85e11e,1929-08-09 09:09:30,2005-08-03 08:25:30,111
4,03623a36-9e65-0980-164f-ffa841234b12,1945-10-04 07:01:26,2026-01-11 12:01:26,48



2. Solución con Window Functions


,PATIENT,primera_visita,ultima_visita,total_visitas
0,00dbff13-8767-26c9-5315-d938617e3e07,2016-05-15 21:30:33,2026-06-07 21:30:33,36
1,012acae3-d0f3-3493-b3cb-bb430437d0f8,2019-11-30 07:49:09,2025-11-15 07:49:09,18
2,0276003c-ce0c-a4cb-b2ed-5443d03cdf46,1997-02-20 03:02:44,2025-01-30 03:02:44,16
3,032e8472-885d-73e5-f9cf-e9505c85e11e,1929-08-09 09:09:30,2005-08-03 08:25:30,111
4,03623a36-9e65-0980-164f-ffa841234b12,1945-10-04 07:01:26,2026-01-11 12:01:26,48



¿Los resultados son exactamente iguales? True


La trampa de LAST_VALUE:
El frame (ventana) por defecto en SQL cuando usas `ORDER BY` dentro de la cláusula `OVER()` es `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. Es decir, la ventana termina en la fila actual. Si no se modifica, `LAST_VALUE` simplemente te devolverá la fecha de la fila que está evaluando en ese instante, no la última fecha de todo el paciente. Para corregirlo, fue obligatorio extender la ventana explícitamente con `RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.

¿Qué enfoque prefiero y por qué?
Prefiero el GROUP BY, porque colapsa naturalmente las filas, lo cual es exactamente el objetivo: una fila por paciente. Usar funciones de ventana aquí es forzar la herramienta: tuvimos que calcular los valores agregados repetidamente fila por fila para luego usar un `DISTINCT` artificial que eliminara los duplicados. Las Window Functions brillan cuando necesitamos mantener el detalle original de las filas, pero para una simple agregación total, GROUP BY es más limpio, rápido y legible.